# 🛒 Superstore Sales Analysis
**Author:** Julian Camilo Torres Rodríguez  
**Tools:** Python · Pandas · PostgreSQL · Matplotlib · Seaborn  
**Dataset:** Sample Superstore (Kaggle)

---
## Objective
Analyze sales data to identify top-performing categories, regional trends, and profitability insights that drive business decisions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
COLORS = {'primary': '#2563EB', 'success': '#16A34A', 'danger': '#DC2626', 'warning': '#D97706'}

print('✅ Libraries loaded')

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('../data/superstore.csv', encoding='latin-1')

# Parse dates
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])
df['Year']       = df['Order Date'].dt.year
df['Month']      = df['Order Date'].dt.to_period('M')

print(f'Shape: {df.shape}')
print(f'Date range: {df["Order Date"].min().date()} → {df["Order Date"].max().date()}')
print(f'Null values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df.head(3)

## 2. Sales & Profit by Category

In [ ]:
cat_summary = df.groupby('Category').agg(
    Total_Sales   = ('Sales',    'sum'),
    Total_Profit  = ('Profit',   'sum'),
    Total_Orders  = ('Order ID', 'nunique'),
    Avg_Discount  = ('Discount', 'mean')
).reset_index()
cat_summary['Profit_Margin_%'] = (cat_summary['Total_Profit'] / cat_summary['Total_Sales'] * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Sales & Profit by Category', fontsize=14, fontweight='bold')

# Sales bar
bars = axes[0].bar(cat_summary['Category'], cat_summary['Total_Sales'] / 1000,
                   color=[COLORS['primary'], COLORS['success'], COLORS['warning']])
axes[0].set_title('Total Sales (USD thousands)')
axes[0].set_ylabel('Sales ($K)')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                 f'${bar.get_height():.0f}K', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Profit margin
colors_margin = [COLORS['success'] if x > 10 else COLORS['warning'] for x in cat_summary['Profit_Margin_%']]
axes[1].barh(cat_summary['Category'], cat_summary['Profit_Margin_%'], color=colors_margin)
axes[1].set_title('Profit Margin (%)')
axes[1].set_xlabel('Margin %')
axes[1].axvline(x=10, color='gray', linestyle='--', alpha=0.5, label='10% threshold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../visualizations/01_category_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(cat_summary.to_string(index=False))

## 3. Regional Performance

In [ ]:
region_summary = df.groupby('Region').agg(
    Sales   = ('Sales',  'sum'),
    Profit  = ('Profit', 'sum'),
    Orders  = ('Order ID', 'nunique')
).reset_index().sort_values('Sales', ascending=False)
region_summary['Margin_%'] = (region_summary['Profit'] / region_summary['Sales'] * 100).round(2)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(region_summary))
width = 0.35

b1 = ax.bar(x - width/2, region_summary['Sales'] / 1000,  width, label='Sales ($K)',  color=COLORS['primary'], alpha=0.85)
b2 = ax.bar(x + width/2, region_summary['Profit'] / 1000, width, label='Profit ($K)', color=COLORS['success'], alpha=0.85)

ax.set_title('Sales vs Profit by Region', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(region_summary['Region'])
ax.set_ylabel('USD (thousands)')
ax.legend()

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
            f'${bar.get_height():.0f}K', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../visualizations/02_regional_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print(region_summary.to_string(index=False))

## 4. Monthly Sales Trend

In [ ]:
monthly = df.groupby(['Year', 'Month'])['Sales'].sum().reset_index()
monthly['Month_str'] = monthly['Month'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
for year in sorted(monthly['Year'].unique()):
    subset = monthly[monthly['Year'] == year]
    ax.plot(subset['Month_str'], subset['Sales'] / 1000,
            marker='o', markersize=4, label=str(year), linewidth=2)

ax.set_title('Monthly Sales Trend by Year', fontsize=13, fontweight='bold')
ax.set_ylabel('Sales ($K)')
ax.set_xlabel('Month')
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Year')
plt.tight_layout()
plt.savefig('../visualizations/03_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Discount vs Profit Correlation

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
scatter = ax.scatter(df['Discount'], df['Profit'],
                     c=df['Sales'], cmap='viridis', alpha=0.4, s=20)
plt.colorbar(scatter, ax=ax, label='Sales ($)')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.6, label='Break-even')
ax.set_title('Discount vs Profit (colored by Sales volume)', fontsize=13, fontweight='bold')
ax.set_xlabel('Discount Rate')
ax.set_ylabel('Profit ($)')
ax.legend()
corr = df[['Discount', 'Profit']].corr().iloc[0, 1]
ax.text(0.7, 0.92, f'Correlation: {corr:.2f}', transform=ax.transAxes,
        fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.savefig('../visualizations/04_discount_profit.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Key Business Insights

In [ ]:
print('=' * 55)
print('         KEY BUSINESS INSIGHTS')
print('=' * 55)

top_cat    = cat_summary.loc[cat_summary['Total_Sales'].idxmax(), 'Category']
best_margin = cat_summary.loc[cat_summary['Profit_Margin_%'].idxmax()]
worst_margin = cat_summary.loc[cat_summary['Profit_Margin_%'].idxmin()]
top_region  = region_summary.iloc[0]
corr_val    = df[['Discount', 'Profit']].corr().iloc[0, 1]
loss_pct    = (df[df['Profit'] < 0].shape[0] / df.shape[0] * 100)

print(f'\n1. Top category by sales: {top_cat}')
print(f'2. Best margin: {best_margin["Category"]} ({best_margin["Profit_Margin_%"]}%)')
print(f'3. Worst margin: {worst_margin["Category"]} ({worst_margin["Profit_Margin_%"]}%)')
print(f'4. Leading region: {top_region["Region"]} (${top_region["Sales"]:,.0f} sales)')
print(f'5. Discount-Profit correlation: {corr_val:.2f} (negative = discounts hurt profit)')
print(f'6. {loss_pct:.1f}% of orders generated a loss — review discount policy')
print('=' * 55)

## Summary
| Insight | Finding |
|---|---|
| Top revenue category | Technology |
| Most profitable category | Technology |
| Least profitable category | Furniture |
| Best region | West |
| Key risk | High discounts strongly reduce profit |

---
*Analysis by Julian Camilo Torres · github.com/juliancamilo*